In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
full_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(full_path)
df

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time distribution')
plt.xlabel('target')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df.head()

In [ ]:
# Task 1: Write your code here:
df.drop(["Order_ID"], axis=1, inplace=True)

In [ ]:
# Task 2: Write your code here:
# Let us see the proportion of each feature
# Analyze missing values
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
# And this is the total number of each one
df.isnull().sum()

In [ ]:
# Let us start with filling the categories features
# Fill categorical columns with 'unknown' - missing likely means "not specified"
df_clean = df.copy()
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

df_clean.isnull().sum()

In [ ]:
# Now it is time to work on each numerical feature # for me I prefer to give it each missing with the mean
for col in ['Courier_Experience_yrs', 'Delivery_Time']:
    df_clean[col] = df_clean[col].fillna(df[col].mean())

df_clean.isnull().sum()

In [ ]:
# Task 3: Write your code here:
df_clean.duplicated().sum() # there are 557 dubs

In [ ]:
df_clean = df_clean.drop_duplicates()
df_clean.duplicated().sum()

In [ ]:
df_clean.head()

In [ ]:
# Task 4: Write your code here:
# It is time to label encode all of the categories as LabelEncoder # except one feature "Traffic_Level" that is order and it is best to make it OneHot # but I couldn't do it due to an error please if you are reading this try to do it and you will understand me

# Encode categorical columns - converts text to integers
le = LabelEncoder()
categorical_cols = ['Weather', 'Time_of_Day', 'Vehicle_Type', "Traffic_Level"]
for col in categorical_cols:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:
X = df_clean.drop(["Delivery_Time"], axis=1) # we must not include the target
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_scaled # is is scaled now

In [ ]:
# Task 6: Write your code here:
df_clean["Delivery_Time"].value_counts()


Delivery_Time = df_clean["Delivery_Time"].value_counts()


plt.figure(figsize=(10, 5))
plt.bar(Delivery_Time.index, Delivery_Time.values, color='coral')
plt.title('Delivery_Time Distribution')
plt.xlabel('Condition')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.grid()
plt.show()

# We can say that it is imbalanced due to the different scale here (around the 60)

In [ ]:
# Task 1: Write your code here:
# X_scaled already did it before in part two
y = df_clean["Delivery_Time"].to_numpy()

In [ ]:
X_scaled[:5], y[:5]

In [ ]:
# Task 2,3,4,5: Write your code here:
# Define Model
model = RandomForestRegressor()

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []
i = 0
for train_idx, test_idx in kf.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))

    # Print Evaluation Metrics
    print(f"K-Fold {i + 1}")
    print(f"MAE {mae_scores[i]}")
    print("-"*40)
    i+=1


# Print Evaluation Metrics
print("\nModel Evaluation Metrics (K-Fold)\n" + "-"*40)
print(f"MAE : {np.mean(mae_scores):.2f}")
print("-"*40)

In [ ]:
df_clean.columns

In [ ]:
# Task 1: Write your code here:
# Feature importance

# must have those:
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']

# + model (sklearn)

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
y_pred = model.predict(X_test)
# len(y_pred), len(y_test) same len


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')
plt.title('y_pred distribution')
plt.xlabel('target')
plt.ylabel('Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(y_test, bins=50, edgecolor='black')
plt.title('y_test distribution')
plt.xlabel('target')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: